# Plant-disease classification — reproducible experiment pipeline

**STATUS: NOT EXECUTED — RESULTS REQUIRED.** This cleaned notebook contains the analysis and training code, but its outputs have been cleared so that stale results from another run are not mistaken for results of this pipeline. Run cells from top to bottom in a configured environment. Do not quote any metric until the cell has run and its output artefacts have been checked.

**Relationship to the revised manuscript**

- The default dataset variant is `segmented`, matching the initial run described in the revised draft. You may override it with `PV_VARIANT`, but the chosen variant must be recorded and reported.
- The pipeline uses a group-aware split. If verified PlantVillage leaf metadata is unavailable or has inadequate coverage, it falls back to near-duplicate groups and explicitly does **not** claim physical-leaf disjointness.
- Model metrics are computed from saved per-image predictions. No random-label metrics, hard-coded baseline results, or unmeasured latency values are used.
- Multi-seed training, baselines, model export/benchmarks, and external validation remain experiments to execute. The external-validation section now records `NOT RUN` when no justified label mapping is supplied and allows later cells to continue.

**Recommended environment:** Kaggle with PlantVillage attached as input. GPU T4 x2 enables parallel seed runs; one GPU and CPU modes are also supported. Installations and data are environment-dependent. The notebook does not bundle the dataset or model weights.


**Code fixes in this copy**

- Near-duplicate detection computes Hamming distances in bounded blocks rather than allocating a full class-by-class distance matrix.
- Training reruns use a fresh timestamped run tag when old seed folders exist, preserving prior checkpoints and avoiding the `FileExistsError` that previously stopped Section 6.
- Section 7 checks that Section 6 produced valid checkpoints and emits a clear error if not.
- Removed an unrelated repeating-alarm / infinite-loop cell from the end of the notebook.


## Kaggle setup: GPU T4 x2 (or single GPU / CPU)

**Notebook settings:** Attach the PlantVillage dataset, set Accelerator = *GPU T4 x2* for parallel seed runs, and enable Internet for package/model-weight downloads. The dataset path is auto-detected; `PV_ROOT` overrides the dataset root (the directory that contains `color/`, `segmented/`, or `grayscale/`). The default `PV_VARIANT` is `segmented`, matching the revised manuscript. Set `PV_LEAF_MAP` and `LEAF_MAP_MODE` only after checking the metadata file's schema.

**Two-GPU behavior:** `GPU_MODE=parallel_seeds` runs independent seeds one per visible GPU; `ddp` uses both GPUs for each seed; `single` uses one GPU, or CPU if no GPU is available. The selected mode is written to `runs/training_mode.json`. Baselines run sequentially on the first available GPU (or CPU); they are not distributed across both GPUs.

Symlinks are created in a scratch layout instead of copying all images. If you rerun into an existing output directory, remove or change `OUT_DIR` first so old model runs cannot be mistaken for new ones.


In [1]:
# Kaggle one-time setup. Restart the kernel if pip reports that an already-imported package was replaced.
!pip uninstall -y -q onnxruntime
!pip install -q ultralytics onnx onnxslim onnxruntime-gpu
!nvidia-smi --query-gpu=index,name,memory.total --format=csv
import os
os.environ.setdefault("WANDB_MODE", "disabled")  # avoid logger integrations prompting for a login


index, name, memory.total [MiB]
0, Tesla T4, 15360 MiB
1, Tesla T4, 15360 MiB


'disabled'

## 0. Configuration and environment capture

In [2]:
import os, sys, json, math, time, random, hashlib, platform, datetime, itertools, shutil
from pathlib import Path
import numpy as np
import pandas as pd

def find_pv_root(variant, base=Path("/kaggle/input")):
    """Folder that contains <variant>/<class>/<image>. Kaggle mount paths differ between dataset versions, so search
    /kaggle/input (max 4 levels) instead of hard-coding one. PV_ROOT overrides."""
    if os.environ.get("PV_ROOT"): return Path(os.environ["PV_ROOT"])
    # Also handle a variant directory directly under the base path.
    if (base / variant).is_dir():
        return base
    for pat in ("*/", "*/*/", "*/*/*/", "*/*/*/*/"):
        for d in sorted(base.glob(pat + variant)):
            if d.is_dir() and sum(1 for c in d.iterdir() if c.is_dir()) >= 30: return d.parent
    return base / "plantvillage-dataset"          # validation below gives a clear missing-path error

_VARIANT = os.environ.get("PV_VARIANT", "segmented")
if _VARIANT not in ("color", "segmented", "grayscale"):
    raise ValueError(f"PV_VARIANT must be one of color/segmented/grayscale, got {_VARIANT!r}")
CFG = dict(
    DATA_ROOT = find_pv_root(_VARIANT),
    VARIANT   = _VARIANT,   # 'color' | 'segmented' | 'grayscale'. The ORIGINAL run used 'segmented' (by accident).
    OUT       = Path(os.environ.get("OUT_DIR", "/kaggle/working/experiments")),
    LEAF_MAP  = Path(os.environ.get("PV_LEAF_MAP", "/kaggle/working/PlantVillage-Dataset/leaf-map.json")),  # from github.com/spMohanty/PlantVillage-Dataset
    SPLIT_SEED = 42,                 # seed of the grouped split (documented in the paper)
    TRAIN_SEEDS = [0, 1, 2],         # independent training seeds
    IMGSZ = 224, EPOCHS = 30, BATCH = 64, PATIENCE = 10,
    NEAR_DUP_HAMMING = 4,            # dHash Hamming threshold (of 64 bits) for 'near duplicate'
    TEST_FRACTION_APPROX = 1/7, VAL_FRACTION_APPROX = 1/7,   # StratifiedGroupKFold folds -> ~14.3 % each; actual proportions are reported, not assumed
    BASELINE_EPOCHS = 10,            # explicit optimisation budget for torchvision baselines (document it)
    GPU_MODE = os.environ.get("GPU_MODE", "auto"),   # 'auto' | 'parallel_seeds' (1 seed per GPU) | 'ddp' (both GPUs per run) | 'single'
    WORKERS = min(4, os.cpu_count() or 1),           # Kaggle GPU sessions have 4 vCPUs; split across parallel processes in Section 6
)
try:
    import torch; N_GPU = torch.cuda.device_count()   # does not create a CUDA context
except Exception:
    N_GPU = 0
CFG["N_GPU"] = N_GPU
if CFG["GPU_MODE"] == "auto": CFG["GPU_MODE"] = "parallel_seeds" if N_GPU >= 2 else "single"
assert CFG["GPU_MODE"] in ("parallel_seeds", "ddp", "single"), f"unknown GPU_MODE {CFG['GPU_MODE']}"
if CFG["GPU_MODE"] != "single" and N_GPU < 2:
    print(f"WARNING: GPU_MODE={CFG['GPU_MODE']} needs 2 GPUs but {N_GPU} found - using 'single'."); CFG["GPU_MODE"] = "single"
if N_GPU == 0: print("WARNING: no CUDA GPU visible. Set Settings -> Accelerator -> GPU T4 x2 in Kaggle.")
# Symlink layout goes to /kaggle/temp (not saved with the notebook version) so Kaggle does not try to archive 54k symlinked images from /kaggle/working.
CFG["LAYOUT_DIR"] = Path(os.environ.get("LAYOUT_DIR", "/kaggle/temp/ultralytics_layout" if Path("/kaggle/temp").exists() else str(CFG["OUT"]/"split"/"ultralytics_layout")))
for d in ("audit","split","runs","predictions","metrics","benchmarks","figures","external"):
    (CFG["OUT"]/d).mkdir(parents=True, exist_ok=True)

def set_all_seeds(seed):
    random.seed(seed); np.random.seed(seed)
    try:
        import torch; torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    except Exception: pass

def capture_environment():
    timestamp = datetime.datetime.now(datetime.timezone.utc).isoformat().replace("+00:00", "Z")
    env = dict(timestamp_utc=timestamp, python=sys.version, platform=platform.platform(), cfg={k:str(v) for k,v in CFG.items()})
    for mod in ("numpy","pandas","sklearn","PIL","torch","torchvision","ultralytics","onnx","onnxruntime","tensorrt"):
        try: m = __import__(mod); env[mod] = getattr(m, "__version__", "unknown")
        except Exception as e: env[mod] = f"not importable: {type(e).__name__}"
    try:
        import torch
        env["cuda_available"] = torch.cuda.is_available()
        if torch.cuda.is_available():
            p = torch.cuda.get_device_properties(0); env["gpu"] = p.name; env["gpu_total_mem_MiB"] = p.total_memory//2**20; env["cuda_runtime"] = torch.version.cuda
            env["n_gpu"] = torch.cuda.device_count(); env["gpus"] = [torch.cuda.get_device_properties(i).name for i in range(torch.cuda.device_count())]   # 'gpu' above = device 0 (used for benchmarks)
            env["cudnn"] = torch.backends.cudnn.version(); env["CUDA_VISIBLE_DEVICES"] = os.environ.get("CUDA_VISIBLE_DEVICES")
    except Exception: pass
    env["cpu"] = platform.processor() or "unknown"
    return env

ENV = capture_environment()
json.dump(ENV, open(CFG["OUT"]/"environment.json","w"), indent=2); print(json.dumps(ENV, indent=2))

Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings.
{
  "timestamp_utc": "2026-10-25T14:33:25.428188Z",
  "python": "3.13.15 (main, Aug  6 2026, 11:06:22) [GCC 13.3.0]",
  "platform": "Linux-6.18.55+-x86_64-with-glibc2.39",
  "cfg": {
    "DATA_ROOT": "/kaggle/input/datasets/abdallahalidev/plantvillage-dataset",
    "VARIANT": "segmented",
    "OUT": "/kaggle/working/experiments",
    "LEAF_MAP": "/kaggle/working/PlantVillage-Dataset/leaf-map.json",
    "SPLIT_SEED": "42",
    "TRAIN_SEEDS": "[0, 1, 2]",
    "IMGSZ": "224",
    "EPOCHS": "30",
    "BATCH": "64",
    "PATIENCE": "10",
    "NEAR_DUP_HAMMING": "4",
    "TEST_FRACTION_APPROX": "0.14285714285714285",
    "VAL_FRACTION_APPROX": "0.14285714285714285",
    "BASELINE_EPOCHS": "10",
    "GPU_

## 1. Dataset audit
Goal: verify the variant, image count, class names, readability, exact duplicates and near duplicates **of the variant actually used**. Class names are read from disk, never typed by hand.

In [3]:
from PIL import Image

def list_images(variant_dir):
    """List supported image files one folder below the selected variant directory."""
    rows = []
    for cdir in sorted(p for p in Path(variant_dir).iterdir() if p.is_dir()):
        for f in sorted(cdir.iterdir()):
            if f.is_file() and f.suffix.lower() in (".jpg", ".jpeg", ".png"):
                rows.append(dict(class_name=cdir.name, filename=f.name,
                                 path=str(f.resolve()), size_bytes=f.stat().st_size))
    return pd.DataFrame(rows)

def sha256_file(p, buf=1 << 20):
    h = hashlib.sha256()
    with open(p, "rb") as fh:
        for chunk in iter(lambda: fh.read(buf), b""):
            h.update(chunk)
    return h.hexdigest()

def dhash64(img):
    """64-bit difference hash calculated from a 9x8 grayscale image."""
    g = img.convert("L").resize((9, 8), Image.LANCZOS)
    a = np.asarray(g, dtype=np.int16)
    bits = (a[:, 1:] > a[:, :-1]).flatten()
    return int("".join("1" if b else "0" for b in bits), 2)

VARIANT_DIR = CFG["DATA_ROOT"] / CFG["VARIANT"]
if not VARIANT_DIR.is_dir():
    visible = [p.name for p in CFG["DATA_ROOT"].iterdir()] if CFG["DATA_ROOT"].is_dir() else "<dataset root does not exist>"
    raise FileNotFoundError(f"Variant directory not found: {VARIANT_DIR}. Dataset-root contents: {visible}. Attach PlantVillage or set PV_ROOT/PV_VARIANT.")
df = list_images(VARIANT_DIR)
if df.empty:
    raise ValueError(f"No .jpg/.jpeg/.png images found under {VARIANT_DIR}/<class>/<image>.")
if df.class_name.nunique() < 2:
    raise ValueError(f"Expected multiple class folders under {VARIANT_DIR}; found {df.class_name.nunique()}.")
print("variant:", CFG["VARIANT"], "| images:", len(df), "| classes:", df.class_name.nunique())

readable, widths, heights, modes, sha, dh = [], [], [], [], [], []
for p in df.path:
    try:
        with Image.open(p) as im:
            im.verify()
        with Image.open(p) as im:
            widths.append(im.width); heights.append(im.height); modes.append(im.mode)
            dh.append(dhash64(im)); readable.append(True)
    except Exception:
        widths.append(-1); heights.append(-1); modes.append("ERR"); dh.append(0); readable.append(False)
    sha.append(sha256_file(p))

df["readable"] = readable; df["width"] = widths; df["height"] = heights
df["mode"] = modes; df["sha256"] = sha; df["dhash"] = np.array(dh, dtype=np.uint64)
df.to_csv(CFG["OUT"] / "audit" / "image_inventory.csv", index=False)
class_names = sorted(df.class_name.unique())
audit = dict(
    variant=CFG["VARIANT"], n_images=int(len(df)), n_classes=int(df.class_name.nunique()),
    n_unreadable=int((~df.readable).sum()),
    image_sizes={f"{w}x{h}": int(n) for (w, h), n in df.groupby(["width", "height"]).size().sort_values(ascending=False).head(10).items()},
    modes={str(k): int(v) for k, v in df["mode"].value_counts().items()},
    class_counts={str(k): int(v) for k, v in df.class_name.value_counts().sort_index().items()},
    n_exact_duplicate_files=int(df.duplicated("sha256", keep=False).sum()),
    exact_duplicate_hashes_with_conflicting_labels=int((df[df.duplicated("sha256", keep=False)].groupby("sha256").class_name.nunique() > 1).sum()),
)
with open(CFG["OUT"] / "audit" / "dataset_audit.json", "w") as fh:
    json.dump(audit, fh, indent=2)
with open(CFG["OUT"] / "audit" / "class_mapping.json", "w") as fh:
    json.dump({i: n for i, n in enumerate(class_names)}, fh, indent=2)
print(json.dumps(audit, indent=2)); print(class_names)
if audit["n_unreadable"]:
    raise RuntimeError(f"Found {audit['n_unreadable']} unreadable image(s). They are listed in audit/image_inventory.csv; repair/remove them before splitting and training.")


variant: segmented | images: 54306 | classes: 38
{
  "variant": "segmented",
  "n_images": 54306,
  "n_classes": 38,
  "n_unreadable": 0,
  "image_sizes": {
    "256x256": 54302,
    "324x512": 1,
    "335x512": 1,
    "466x512": 1,
    "470x512": 1
  },
  "modes": {
    "RGB": 54305,
    "L": 1
  },
  "class_counts": {
    "Apple___Apple_scab": 630,
    "Apple___Black_rot": 621,
    "Apple___Cedar_apple_rust": 275,
    "Apple___healthy": 1645,
    "Blueberry___healthy": 1502,
    "Cherry_(including_sour)___Powdery_mildew": 1052,
    "Cherry_(including_sour)___healthy": 854,
    "Corn_(maize)___Cercospora_leaf_spot Gray_leaf_spot": 513,
    "Corn_(maize)___Common_rust_": 1192,
    "Corn_(maize)___Northern_Leaf_Blight": 985,
    "Corn_(maize)___healthy": 1162,
    "Grape___Black_rot": 1180,
    "Grape___Esca_(Black_Measles)": 1384,
    "Grape___Leaf_blight_(Isariopsis_Leaf_Spot)": 1076,
    "Grape___healthy": 423,
    "Orange___Haunglongbing_(Citrus_greening)": 5507,
    "Peach___Bacter

In [4]:
# Near-duplicate clusters (within each class) from dHash Hamming distance <= NEAR_DUP_HAMMING.
# Exact byte-identical files are linked globally, including files assigned different class labels.
POP = np.array([bin(i).count("1") for i in range(256)], dtype=np.uint8)

def hamming_block(h1, h2):
    """Pairwise 64-bit Hamming distances using bounded temporary memory."""
    h1 = np.asarray(h1, dtype=np.uint64)
    h2 = np.asarray(h2, dtype=np.uint64)
    xor = np.bitwise_xor(h1[:, None], h2[None, :])
    dist = np.zeros(xor.shape, dtype=np.uint8)
    # Count set bits a byte at a time instead of materialising an 8x-expanded array.
    for shift in range(0, 64, 8):
        byte = ((xor >> np.uint64(shift)) & np.uint64(0xff)).astype(np.uint8)
        dist += POP[byte]
    return dist

class DSU:
    def __init__(self, n): self.p = list(range(n))
    def f(self, a):
        while self.p[a] != a:
            self.p[a] = self.p[self.p[a]]; a = self.p[a]
        return a
    def u(self, a, b): self.p[self.f(a)] = self.f(b)

def near_duplicate_clusters(frame, thr, chunk_size=256):
    """Return cluster IDs without allocating a full pairwise matrix.

    Difference-hash comparisons are restricted to images within the same class.
    Exact SHA-256 matches are merged globally, including across conflicting labels.
    """
    frame = frame.reset_index(drop=True)
    dsu = DSU(len(frame))
    for _, sub in frame.groupby("class_name", sort=False):
        pos = sub.index.to_numpy()
        hashes = sub.dhash.to_numpy(dtype=np.uint64)
        n = len(hashes)
        for begin in range(0, n, chunk_size):
            stop = min(begin + chunk_size, n)
            distances = hamming_block(hashes[begin:stop], hashes)
            # Keep each undirected edge once, excluding self-links and lower triangle.
            row_ids = np.arange(begin, stop)[:, None]
            col_ids = np.arange(n)[None, :]
            distances[row_ids >= col_ids] = 255
            ii, jj = np.where(distances <= thr)
            for a, b in zip(ii, jj):
                dsu.u(int(pos[begin + a]), int(pos[b]))
    first_hash = {}
    for i, digest in enumerate(frame.sha256.astype(str)):
        if digest in first_hash:
            dsu.u(i, first_hash[digest])
        else:
            first_hash[digest] = i
    root_to_id = {}
    ids = np.empty(len(frame), dtype=np.int64)
    for i in range(len(frame)):
        root = dsu.f(i)
        if root not in root_to_id: root_to_id[root] = len(root_to_id)
        ids[i] = root_to_id[root]
    return ids

df["near_dup_cluster"] = near_duplicate_clusters(df, CFG["NEAR_DUP_HAMMING"])
cluster_sizes = df.groupby("near_dup_cluster").size()
conflicting_exact = df[df.duplicated("sha256", keep=False)].groupby("sha256").class_name.nunique()
print("clusters:", len(cluster_sizes), "| clusters with >1 image:", int((cluster_sizes > 1).sum()),
      "| images in multi-image clusters:", int(cluster_sizes[cluster_sizes > 1].sum()),
      "| exact duplicate hashes with conflicting labels:", int((conflicting_exact > 1).sum()))
df.to_csv(CFG["OUT"] / "audit" / "image_inventory.csv", index=False)


clusters: 52224 | clusters with >1 image: 1058 | images in multi-image clusters: 3140 | exact duplicate hashes with conflicting labels: 0


## 2. Leaf-identity / grouping metadata
The official repository (github.com/spMohanty/PlantVillage-Dataset) contains a `leaf_grouping/` directory and `leaf-map.json`, and states that the pre-defined split keeps images of the same leaf together. **The schema of these files was not inspected during the audit.** Run `inspect_leaf_map` first, choose the matching adapter, and let the coverage assertion decide whether the metadata is usable. If it is not, the code falls back to near-duplicate clusters and records that fact (`GROUP_SOURCE`) - the manuscript must then say so.

In [5]:
def inspect_leaf_map(path, n=5):
    with open(path, encoding="utf-8") as fh:
        obj = json.load(fh)
    print("top-level type:", type(obj).__name__, "| length:", len(obj) if hasattr(obj, "__len__") else "n/a")
    if isinstance(obj, dict):
        for k in list(obj)[:n]: print(repr(k), "->", repr(obj[k])[:200])
    elif isinstance(obj, list):
        for x in obj[:n]: print(repr(x)[:200])
    return obj

def groups_from_leaf_map(frame, obj, mode):
    """Adapt an inspected dict: file2group={file/group/path: group} or group2files={group: [files]}.

    Path-like keys are matched by class/filename or full path. Basename matching is used only when unique,
    so duplicate basenames cannot silently receive another file's leaf ID.
    """
    def norm(value): return str(value).replace("\\", "/").lstrip("./")
    raw = {}
    if not isinstance(obj, dict):
        raise TypeError("Expected a JSON object/dict; inspect and add an explicit adapter for this schema")
    if mode == "file2group":
        for k, group in obj.items(): raw[norm(k)] = str(group)
    elif mode == "group2files":
        for group, files in obj.items():
            if not isinstance(files, (list, tuple, set)):
                raise TypeError(f"group2files value for {group!r} is not a list")
            for file_key in files: raw[norm(file_key)] = str(group)
    else:
        raise ValueError("LEAF_MAP_MODE must be 'file2group' or 'group2files'")
    by_basename = {}
    for key, group in raw.items(): by_basename.setdefault(Path(key).name, set()).add(group)
    out = []
    for row in frame.itertuples():
        rel = norm(f"{row.class_name}/{row.filename}")
        full = norm(row.path)
        candidates = (rel, full, norm(row.filename))
        group = next((raw[k] for k in candidates if k in raw), None)
        if group is None:
            vals = by_basename.get(Path(row.filename).name, set())
            if len(vals) == 1: group = next(iter(vals))
        out.append(group)
    return pd.Series(out, index=frame.index, dtype="object")

GROUP_SOURCE = None
if CFG["LEAF_MAP"].is_file():
    leaf_obj = inspect_leaf_map(CFG["LEAF_MAP"])
    LEAF_MAP_MODE = os.environ.get("LEAF_MAP_MODE")  # set only after inspecting the printed JSON sample
    if LEAF_MAP_MODE:
        try:
            g = groups_from_leaf_map(df, leaf_obj, LEAF_MAP_MODE)
            coverage = float(g.notna().mean())
            print(f"leaf-map coverage: {coverage:.4f}")
            if coverage >= 0.995:
                # Unmapped files (up to the 0.5% tolerance) receive unique provisional IDs;
                # the subsequent near-duplicate merge can still link them to any mapped copy.
                df["group_id"] = [
                    f"leaf:{row.class_name}:{value}" if pd.notna(value) else f"leaf-unmapped:{i}"
                    for i, (row, value) in enumerate(zip(df.itertuples(), g.to_numpy()))
                ]
                GROUP_SOURCE = "PlantVillage leaf-map.json"
            else:
                print("Leaf-map coverage below 99.5%; falling back to near-duplicate clusters.")
        except (TypeError, ValueError, KeyError) as exc:
            print(f"Could not adapt leaf-map schema ({type(exc).__name__}: {exc}); falling back to near-duplicate clusters.")
    else:
        print("LEAF_MAP_MODE is unset; inspect the JSON and set it only if an adapter matches. Falling back to near-duplicate clusters.")
else:
    print("leaf-map.json not found at", CFG["LEAF_MAP"], "- falling back to near-duplicate clusters.")

if GROUP_SOURCE is None:
    df["group_id"] = "nd:" + df.near_dup_cluster.astype(str)
    GROUP_SOURCE = "near-duplicate clustering (dHash+SHA-256), NOT leaf identity"
else:
    # Merge physical-leaf groups and near-duplicate groups transitively.
    d = DSU(len(df)); first = {}
    for i, (leaf_group, dup_group) in enumerate(zip(df.group_id, df.near_dup_cluster)):
        for key in (("leaf", leaf_group), ("duplicate", int(dup_group))):
            if key in first: d.u(i, first[key])
            else: first[key] = i
    df["group_id"] = ["merged:" + str(d.f(i)) for i in range(len(df))]
    GROUP_SOURCE += " + near-duplicate clusters merged"
if df.group_id.isna().any(): raise ValueError("Grouping produced missing group IDs; inspect leaf metadata adapter.")
print("GROUP_SOURCE =", GROUP_SOURCE, "| n_groups =", df.group_id.nunique())


leaf-map.json not found at /kaggle/working/PlantVillage-Dataset/leaf-map.json - falling back to near-duplicate clusters.
GROUP_SOURCE = near-duplicate clustering (dHash+SHA-256), NOT leaf identity | n_groups = 52224


## 3. Group-aware, class-stratified split (train / val / test)

In [6]:
from sklearn.model_selection import StratifiedGroupKFold

def make_split(frame, seed, n_test_folds=7, n_val_folds=6):
    y = frame.class_name.to_numpy(); g = frame.group_id.to_numpy(); idx = np.arange(len(frame))
    s1 = StratifiedGroupKFold(n_splits=n_test_folds, shuffle=True, random_state=seed)
    trainval, test = next(s1.split(idx, y, g))
    s2 = StratifiedGroupKFold(n_splits=n_val_folds, shuffle=True, random_state=seed+1)
    tr, va = next(s2.split(trainval, y[trainval], g[trainval]))
    split = np.empty(len(frame), dtype=object)
    split[trainval[tr]] = "train"; split[trainval[va]] = "val"; split[test] = "test"
    return split

try:
    df["split"] = make_split(df, CFG["SPLIT_SEED"])
except ValueError as exc:
    raise ValueError("Could not create a class-stratified group split. Check group metadata, class/group counts, and scikit-learn version; do not switch to an image-random split silently.") from exc

# --- leakage assertions -------------------------------------------------------------------
grp_split = df.groupby("group_id").split.nunique(); assert (grp_split==1).all(), "A group occurs in >1 partition"
dup = df[df.duplicated("sha256", keep=False)]
assert (dup.groupby("sha256").split.nunique()==1).all() if len(dup) else True, "Exact duplicate across partitions"
cls_split = df.groupby(["class_name","split"]).size().unstack(fill_value=0)
assert (cls_split>0).all().all(), f"A class is missing from a partition:\n{cls_split[(cls_split==0).any(axis=1)]}"

report = dict(split_seed=CFG["SPLIT_SEED"], group_source=GROUP_SOURCE, n_groups=int(df.group_id.nunique()),
              counts=df.split.value_counts().to_dict(),
              proportions={k: round(v/len(df),4) for k,v in df.split.value_counts().items()},
              groups_per_split=df.groupby("split").group_id.nunique().to_dict(),
              min_images_per_class_per_split=cls_split.min().to_dict(),
              statement="Partitions are disjoint at the level of GROUP_SOURCE. Disjointness at the level of physical leaves is only as good as that metadata.")
json.dump(report, open(CFG["OUT"]/"split"/"split_report.json","w"), indent=2)
df[["path","class_name","filename","sha256","group_id","split"]].to_csv(CFG["OUT"]/"split"/"split_manifest.csv", index=False)
cls_split.to_csv(CFG["OUT"]/"split"/"class_counts_per_split.csv")
SPLIT_MANIFEST_SHA = hashlib.sha256(open(CFG["OUT"]/"split"/"split_manifest.csv","rb").read()).hexdigest()
print(json.dumps(report, indent=2)); print("manifest sha256:", SPLIT_MANIFEST_SHA)

# --- materialise the Ultralytics folder layout with symlinks (no copying) -------------------
SPLIT_DIR = CFG["LAYOUT_DIR"]; SPLIT_DIR.parent.mkdir(parents=True, exist_ok=True)
# Remove only the dedicated layout directory, not its parent or an arbitrary data directory.
_layout_resolved = SPLIT_DIR.resolve()
_out_resolved = CFG["OUT"].resolve()
_data_resolved = CFG["DATA_ROOT"].resolve()
if (_layout_resolved in (Path("/"), Path.home().resolve())
        or _out_resolved == _layout_resolved or _out_resolved.is_relative_to(_layout_resolved)
        or _data_resolved == _layout_resolved or _data_resolved.is_relative_to(_layout_resolved)):
    raise ValueError(f"Unsafe LAYOUT_DIR target: {SPLIT_DIR}. It must be a dedicated scratch folder, not an ancestor of OUT_DIR or the dataset.")
if SPLIT_DIR.is_symlink():
    SPLIT_DIR.unlink()
elif SPLIT_DIR.exists() and SPLIT_DIR.is_dir():
    shutil.rmtree(SPLIT_DIR)
elif SPLIT_DIR.exists():
    SPLIT_DIR.unlink()
for r in df.itertuples():
    dst = SPLIT_DIR / r.split / r.class_name / r.filename
    dst.parent.mkdir(parents=True, exist_ok=True)
    os.symlink(str(Path(r.path).resolve()), dst)
print("layout at", SPLIT_DIR)

{
  "split_seed": 42,
  "group_source": "near-duplicate clustering (dHash+SHA-256), NOT leaf identity",
  "n_groups": 52224,
  "counts": {
    "train": 39048,
    "test": 7697,
    "val": 7561
  },
  "proportions": {
    "train": 0.719,
    "test": 0.1417,
    "val": 0.1392
  },
  "groups_per_split": {
    "test": 7480,
    "train": 37458,
    "val": 7286
  },
  "min_images_per_class_per_split": {
    "test": 22,
    "train": 107,
    "val": 23
  },
  "statement": "Partitions are disjoint at the level of GROUP_SOURCE. Disjointness at the level of physical leaves is only as good as that metadata."
}
manifest sha256: dc9a9780029bd85613574362f4f67ddceb55f4f538b1678382c7ea5cc07ba564
layout at /kaggle/working/experiments/split/ultralytics_layout


## 4. Document the augmentation pipeline that Ultralytics actually builds
The original manuscript listed mosaic and HSV jitter, but the audit could only confirm that those *arguments were set*, not that the classification dataloader applied them. This cell prints the real transform code so the Methods section can be written from it.

In [7]:
import inspect
from ultralytics.data import augment as ua
for _name in ("classify_augmentations", "classify_transforms"):
    _fn = getattr(ua, _name, None)
    if _fn is None:
        print(f"Ultralytics {getattr(ua, '__file__', '<unknown>')} does not expose {_name}; inspect the installed classify loader before describing transforms.")
    else:
        try:
            print(f"\n--- {_name} ---")
            print(inspect.getsource(_fn))
        except (OSError, TypeError) as exc:
            print(f"Could not retrieve source for {_name}: {type(exc).__name__}: {exc}")



--- classify_augmentations ---
def classify_augmentations(
    size: int = 224,
    mean: tuple[float, float, float] = DEFAULT_MEAN,
    std: tuple[float, float, float] = DEFAULT_STD,
    scale: tuple[float, float] | None = None,
    ratio: tuple[float, float] | None = None,
    hflip: float = 0.5,
    vflip: float = 0.0,
    auto_augment: str | None = None,
    hsv_h: float = 0.015,  # image HSV-Hue augmentation (fraction)
    hsv_s: float = 0.4,  # image HSV-Saturation augmentation (fraction)
    hsv_v: float = 0.4,  # image HSV-Value augmentation (fraction)
    force_color_jitter: bool = False,
    erasing: float = 0.0,
    interpolation: str = "BILINEAR",
):
    """Create a composition of image augmentation transforms for classification tasks.

    This function generates a set of image transformations suitable for training classification models. It includes
    options for resizing, flipping, color jittering, auto augmentation, and random erasing. The defaults here are
    standa

## 5. Metric code (computed ONLY from saved predictions) + unit tests on a synthetic toy array
`evaluate_predictions` takes a predictions DataFrame (schema in Section 7). Definitions: Top-k accuracy = fraction of images whose true class is among the k highest scores. Precision/recall/F1 per class from the confusion matrix; **macro** = unweighted mean over the 38 classes, **weighted** = mean weighted by class support. ROC-AUC = one-vs-rest AUC per class, macro-averaged (`sklearn.metrics.roc_auc_score(multi_class="ovr", average="macro")`). mAP = mean over classes of step-wise average precision (`average_precision_score`, no trapezoid interpolation). ECE = 15 equal-width bins on the top-1 confidence. Brier = mean over images of the sum over classes of (p - onehot)^2.

In [8]:
from sklearn.metrics import (precision_recall_fscore_support, confusion_matrix, roc_auc_score, average_precision_score)

def evaluate_predictions(P, class_names, groups=None, n_boot=2000, boot_seed=123, ece_bins=15):
    C = len(class_names); pcols = [f"p_{i:02d}" for i in range(C)]
    if C < 2: raise ValueError("At least two class names are required")
    if P.empty: raise ValueError("Cannot evaluate an empty prediction table")
    missing = [c for c in ["true_idx", "pred_top1_idx", *pcols] if c not in P.columns]
    if missing: raise ValueError(f"Prediction table is missing required columns: {missing}")
    y = P["true_idx"].to_numpy(int); prob = P[pcols].to_numpy(float); yhat = prob.argmax(1)
    # ---- sanity checks (edge cases) ----
    assert prob.ndim == 2 and prob.shape[1] == C and np.isfinite(prob).all(), "invalid probability matrix"
    assert ((y >= 0) & (y < C)).all(), "true class index outside class_names range"
    if groups is not None and len(groups) != len(y): raise ValueError("groups length must match prediction rows")
    assert np.allclose(prob.sum(1), 1, atol=1e-3), "probabilities do not sum to 1"
    assert (prob>=-1e-9).all(), "negative probabilities"
    assert (yhat == P["pred_top1_idx"].to_numpy(int)).all(), "saved top-1 disagrees with argmax of saved scores"
    present = np.unique(y); absent = sorted(set(range(C)) - set(present))
    top5 = np.argsort(-prob, 1)[:, :5]
    out = {"n_images": int(len(y)), "n_classes": C, "classes_absent_from_test": [class_names[i] for i in absent]}
    out["top1_accuracy"] = float((yhat==y).mean()); out["top5_accuracy"] = float((top5==y[:,None]).any(1).mean())
    out["n_top1_errors"] = int((yhat!=y).sum()); out["n_top5_errors"] = int((~(top5==y[:,None]).any(1)).sum())
    labels = list(range(C))
    p_, r_, f_, s_ = precision_recall_fscore_support(y, yhat, labels=labels, zero_division=0)
    per_class = pd.DataFrame(dict(class_idx=labels, class_name=class_names, precision=p_, recall=r_, f1=f_, support=s_))
    for avg in ("macro","weighted"):
        pp, rr, ff, _ = precision_recall_fscore_support(y, yhat, labels=labels, average=avg, zero_division=0)
        out[f"{avg}_precision"], out[f"{avg}_recall"], out[f"{avg}_f1"] = float(pp), float(rr), float(ff)
    cm = confusion_matrix(y, yhat, labels=labels)
    # ---- ROC-AUC (OvR, macro) over classes with both positives and negatives ----
    ok = [c for c in labels if 0 < (y==c).sum() < len(y)]
    aucs = [roc_auc_score((y == c).astype(int), prob[:, c]) for c in ok]
    out["roc_auc_ovr_macro"] = float(np.mean(aucs)) if aucs else float("nan")
    out["roc_auc_classes_used"] = len(ok)
    # ---- mean average precision (macro over classes with positives) ----
    aps = {c: average_precision_score((y == c).astype(int), prob[:, c]) for c in ok}
    per_class["average_precision"] = [aps.get(c, np.nan) for c in labels]
    out["mAP_macro_stepwise_AP"] = float(np.mean(list(aps.values()))) if aps else float("nan")
    # ---- calibration ----
    conf = prob.max(1); correct = (yhat==y).astype(float); edges = np.linspace(0,1,ece_bins+1); ece = 0.0
    for lo,hi in zip(edges[:-1], edges[1:]):
        m = (conf>lo)&(conf<=hi)
        if m.any(): ece += m.mean()*abs(correct[m].mean()-conf[m].mean())
    onehot = np.eye(C)[y]
    out["ece_15bin_top1"] = float(ece); out["brier_multiclass"] = float(((prob-onehot)**2).sum(1).mean())
    out["nll"] = float(-np.log(np.clip(prob[np.arange(len(y)), y], 1e-12, 1)).mean())
    # ---- uncertainty: image-level bootstrap and (if groups given) group-level cluster bootstrap ----
    rng = np.random.default_rng(boot_seed); n = len(y)
    def macro_f1(idx): return precision_recall_fscore_support(y[idx], yhat[idx], labels=labels, average="macro", zero_division=0)[2]
    b_acc, b_f1 = [], []
    for _ in range(n_boot):
        idx = rng.integers(0, n, n); b_acc.append((yhat[idx]==y[idx]).mean()); b_f1.append(macro_f1(idx))
    out["top1_ci95_image_bootstrap"] = [float(np.percentile(b_acc,2.5)), float(np.percentile(b_acc,97.5))]
    out["macro_f1_ci95_image_bootstrap"] = [float(np.percentile(b_f1,2.5)), float(np.percentile(b_f1,97.5))]
    if groups is not None:
        g = np.asarray(groups); ug, inv = np.unique(g, return_inverse=True); members = [np.where(inv==k)[0] for k in range(len(ug))]
        b = []
        for _ in range(n_boot):
            pick = rng.integers(0, len(ug), len(ug)); idx = np.concatenate([members[k] for k in pick]); b.append((yhat[idx]==y[idx]).mean())
        out["top1_ci95_group_bootstrap"] = [float(np.percentile(b,2.5)), float(np.percentile(b,97.5))]
    return out, per_class, cm

def predictions_frame_from_arrays(y_true, prob, class_names, ids=None):
    C = len(class_names); df_ = pd.DataFrame(prob, columns=[f"p_{i:02d}" for i in range(C)])
    df_.insert(0, "true_idx", y_true); df_.insert(1, "pred_top1_idx", prob.argmax(1))
    if ids is not None: df_.insert(0, "image_id", ids)
    return df_

**Unit tests - SYNTHETIC TOY DATA, verifies the metric code only. These numbers are NOT experimental results and must never appear in the paper.**

In [9]:
def _toy_tests():
    rng = np.random.default_rng(0); C = 5; names = [f"c{i}" for i in range(C)]
    y = np.repeat(np.arange(C), 40)
    # (1) perfect classifier -> every metric = 1, ECE ~ small, Brier ~ 0
    prob = np.eye(C)[y]*0.98 + 0.005; prob /= prob.sum(1, keepdims=True)
    o,_,cm = evaluate_predictions(predictions_frame_from_arrays(y, prob, names), names, n_boot=50)
    assert o["top1_accuracy"]==1 and o["macro_f1"]==1 and abs(o["roc_auc_ovr_macro"]-1)<1e-9 and abs(o["mAP_macro_stepwise_AP"]-1)<1e-9 and np.trace(cm)==len(y)
    # (2) always predict class 0 -> top1 = 1/C, class-0 recall = 1, others 0
    prob = np.tile(np.eye(C)[0]*0.96+0.01, (len(y),1)); prob/=prob.sum(1,keepdims=True)
    o,pc,_ = evaluate_predictions(predictions_frame_from_arrays(y, prob, names), names, n_boot=50)
    assert abs(o["top1_accuracy"]-1/C)<1e-12 and pc.recall.iloc[0]==1 and pc.recall.iloc[1:].sum()==0
    # (3) known confusion: 100 images, 2 classes wrong in a controlled way
    y3 = np.array([0]*50+[1]*50); p3 = np.zeros((100,2)); p3[:,0]=0.9; p3[:,1]=0.1; p3[50:,:]=[0.1,0.9]; p3[50:55,:]=[0.9,0.1]   # 5 errors in class 1
    o,pc,cm = evaluate_predictions(predictions_frame_from_arrays(y3, p3, ["a","b"]), ["a","b"], n_boot=50)
    assert cm.tolist()==[[50,0],[5,45]] and abs(o["top1_accuracy"]-0.95)<1e-12
    assert abs(pc.precision.iloc[0]-50/55)<1e-12 and abs(pc.recall.iloc[1]-0.9)<1e-12
    # (4) invalid probabilities are rejected
    bad = prob.copy(); bad[0,0] = np.nan
    try: evaluate_predictions(predictions_frame_from_arrays(y, bad, names), names, n_boot=5); raise SystemExit("NaN accepted")
    except AssertionError: pass
    # (5) class absent from labels is reported, not silently scored
    y5 = np.array([0,0,1,1,1]); p5 = np.eye(3)[y5]*0.9+0.05; p5/=p5.sum(1,keepdims=True)
    o,_,_ = evaluate_predictions(predictions_frame_from_arrays(y5, p5, ["a","b","c"]), ["a","b","c"], n_boot=5)
    assert o["classes_absent_from_test"]==["c"] and o["roc_auc_classes_used"]==2
    print("metric unit tests passed")
_toy_tests()

metric unit tests passed


## 6. Training YOLOv8n-cls (several independent seeds)
Settings mirror the original run so that the comparison is clean: `yolov8n-cls.pt`, 30 epochs, 224 px, batch 64, patience 10, `optimizer="auto"` (resolved to MuSGD lr=0.01, momentum 0.9 in the original log), **`cos_lr=False`** (the original run used Ultralytics' default linear decay, not cosine). Only the `val` split is used during training and for checkpoint selection (`best.pt`). Record the resolved optimiser line from the log in the paper.

**Kaggle T4 x2 - `CFG["GPU_MODE"]`** (env `GPU_MODE`, default `auto`):
- `parallel_seeds` (default with 2 GPUs): one independent single-GPU training run per GPU, launched as separate processes (`CUDA_VISIBLE_DEVICES`). Each run is the same computation as `single`, so the settings above are unchanged; seeds 0 and 1 run together, seed 2 follows.
- `ddp`: every run uses both GPUs (Ultralytics DDP, `batch=64` is the *total*, 32 per GPU). Faster per run, but BatchNorm statistics and data order differ from a single-GPU run, so results are **not** comparable with the original single-GPU run. Report the mode.
- `single`: GPU 0 only.
The chosen mode is saved to `runs/training_mode.json` and into every metrics file.


In [10]:
from ultralytics import YOLO
import subprocess, textwrap

# Pre-download pretrained weights once, so parallel workers do not race on the same download (needs Internet = On or the files in the cwd).
for _w in ("yolov8n-cls.pt", "yolo11n.pt"):          # 2nd file: used by Ultralytics' AMP self-check
    try: YOLO(_w)
    except Exception as e: print("could not pre-fetch", _w, "->", type(e).__name__, e)

def train_one(seed, data_dir=SPLIT_DIR, tag="yolov8n_cls", device=None):
    """Train one seed. A device list invokes DDP; if no CUDA device exists, use CPU (slow)."""
    set_all_seeds(seed)
    if device is None: device = 0 if CFG["N_GPU"] else "cpu"
    workers = CFG["WORKERS"]
    if isinstance(device, (list, tuple)): workers = max(1, workers // len(device))
    run_name = f"{tag}_seed{seed}"
    expected_dir = CFG["OUT"] / "runs" / run_name
    if expected_dir.exists():
        raise FileExistsError(f"Run directory already exists: {expected_dir}. Set OUT_DIR to a fresh folder or remove this prior run before training.")
    model = YOLO("yolov8n-cls.pt")
    model.train(data=str(data_dir), epochs=CFG["EPOCHS"], imgsz=CFG["IMGSZ"], batch=CFG["BATCH"], patience=CFG["PATIENCE"],
                seed=seed, deterministic=True, pretrained=True, cos_lr=False, optimizer="auto", workers=workers, device=device,
                project=str(CFG["OUT"] / "runs"), name=run_name, exist_ok=False, plots=True, verbose=True)
    return CFG["OUT"]/"runs"/f"{tag}_seed{seed}"

# Worker for 'parallel_seeds': identical settings to train_one(device=0), run in a fresh process that sees exactly one GPU.
WORKER = CFG["OUT"]/"runs"/"_train_worker.py"
WORKER.write_text(textwrap.dedent('''\
    import sys, json, random
    import numpy as np, torch
    from ultralytics import YOLO
    a = json.loads(sys.argv[1]); s = a["seed"]
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)
    YOLO("yolov8n-cls.pt").train(data=a["data"], epochs=a["epochs"], imgsz=a["imgsz"], batch=a["batch"], patience=a["patience"],
                seed=s, deterministic=True, pretrained=True, cos_lr=False, optimizer="auto", workers=a["workers"], device=0,
                project=a["project"], name=a["name"], exist_ok=False, plots=True, verbose=True)
'''))

def train_seeds_parallel(seeds, tag="yolov8n_cls", stagger_s=20, poll_s=15):
    """One seed per GPU; when a GPU frees up, the next pending seed starts on it. Output of each run -> runs/<name>_stdout.log"""
    per_proc_workers = max(1, CFG["WORKERS"] // CFG["N_GPU"])
    for s in seeds:
        target = CFG["OUT"] / "runs" / f"{tag}_seed{s}"
        if target.exists():
            raise FileExistsError(f"Run directory already exists: {target}. Set OUT_DIR to a fresh folder or remove this prior run before training.")
    pending, running = list(seeds), {}
    def kill_all():
        for _, (_, p, lf) in running.items(): p.kill(); lf.close()
    while pending or running:
        for gpu in range(CFG["N_GPU"]):
            if gpu not in running and pending:
                s = pending.pop(0); name = f"{tag}_seed{s}"
                args = dict(seed=s, data=str(SPLIT_DIR), epochs=CFG["EPOCHS"], imgsz=CFG["IMGSZ"], batch=CFG["BATCH"], patience=CFG["PATIENCE"],
                            workers=per_proc_workers, project=str(CFG["OUT"]/"runs"), name=name)
                lf = open(CFG["OUT"]/"runs"/f"{name}_stdout.log", "w")
                p = subprocess.Popen([sys.executable, str(WORKER), json.dumps(args)], stdout=lf, stderr=subprocess.STDOUT,
                                     env={**os.environ, "CUDA_VISIBLE_DEVICES": str(gpu)})
                running[gpu] = (s, p, lf); print(f"seed {s} -> GPU {gpu} (pid {p.pid})"); time.sleep(stagger_s)
        for gpu, (s, p, lf) in list(running.items()):
            rc = p.poll()
            if rc is None: continue
            lf.close(); del running[gpu]
            if rc != 0:
                kill_all(); tail = (CFG["OUT"]/"runs"/f"{tag}_seed{s}_stdout.log").read_text()[-3000:]
                raise RuntimeError(f"seed {s} on GPU {gpu} exited with code {rc}. Log tail:\n{tail}")
            print("finished seed", s, "on GPU", gpu)
        if running or pending: time.sleep(poll_s)
    return {s: CFG["OUT"]/"runs"/f"{tag}_seed{s}" for s in seeds}

# ---- run ----
# Keep previous outputs intact. If any expected seed folder exists, select a fresh tag
# for this run so stale checkpoints cannot be mistaken for the current experiment.
def choose_training_tag(base_tag, seeds):
    runs_root = CFG["OUT"] / "runs"
    candidate = base_tag
    if any((runs_root / f"{candidate}_seed{s}").exists() for s in seeds):
        stamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
        candidate = f"{base_tag}_{stamp}"
        suffix = 1
        while any((runs_root / f"{candidate}_seed{s}").exists() for s in seeds):
            candidate = f"{base_tag}_{stamp}_{suffix}"
            suffix += 1
        print(f"Existing seed run folder(s) found. Preserving them and starting a new run with tag: {candidate}")
    return candidate

RUN_DIRS = {}
RUN_TAG = choose_training_tag("yolov8n_cls", CFG["TRAIN_SEEDS"])
json.dump(dict(gpu_mode=CFG["GPU_MODE"], n_gpu=CFG["N_GPU"], workers_total=CFG["WORKERS"], seeds=CFG["TRAIN_SEEDS"], run_tag=RUN_TAG), open(CFG["OUT"]/"runs"/"training_mode.json","w"), indent=2)
if CFG["GPU_MODE"] == "parallel_seeds":
    RUN_DIRS = train_seeds_parallel(CFG["TRAIN_SEEDS"], tag=RUN_TAG)
else:
    for s in CFG["TRAIN_SEEDS"]:
        _device = [0, 1] if CFG["GPU_MODE"] == "ddp" else (0 if CFG["N_GPU"] else "cpu")
        RUN_DIRS[s] = train_one(s, tag=RUN_TAG, device=_device)
        print("finished seed", s, "->", RUN_DIRS[s])
# Ultralytics silently increments a run name that already exists (…seed0 -> …seed02) - make sure we point at the real outputs.
for s, rd in RUN_DIRS.items(): assert (rd/"weights"/"best.pt").exists(), f"no best.pt for seed {s} in {rd} (stale/renamed run folder?)"


seed 0 -> GPU 0 (pid 167)
seed 1 -> GPU 1 (pid 173)
finished seed 0 on GPU 0
finished seed 1 on GPU 1
seed 2 -> GPU 0 (pid 307)
finished seed 2 on GPU 0


In [11]:
if not globals().get("RUN_DIRS"):
    raise RuntimeError("Training results are unavailable: run Section 6 successfully before Section 7. Check the training logs if Section 6 failed.")
_missing_runs = [str(p) for p in RUN_DIRS.values() if not (Path(p) / "weights" / "best.pt").is_file()]
if _missing_runs:
    raise RuntimeError("Missing trained checkpoint(s); Section 6 did not finish successfully: " + ", ".join(_missing_runs))

def predict_split(weights, split_dir, split="test", batch=64, tag=""):
    model = YOLO(str(weights)); names = [model.names[i] for i in range(len(model.names))]
    assert names == class_names, "model class order differs from dataset class order"
    files = sorted((Path(split_dir) / split).rglob("*.*")); files = [f for f in files if f.suffix.lower() in (".jpg", ".jpeg", ".png")]
    if not files: raise FileNotFoundError(f"No images found for split={split!r} under {Path(split_dir) / split}")
    ids, y_true, probs = [], [], []
    for i in range(0, len(files), 256):
        chunk = files[i:i+256]
        res = model.predict([str(f) for f in chunk], imgsz=CFG["IMGSZ"], batch=batch, verbose=False, half=False)
        for f, r in zip(chunk, res):
            ids.append(str(f.relative_to(split_dir))); y_true.append(class_names.index(f.parent.name)); probs.append(r.probs.data.float().cpu().numpy())
    prob = np.stack(probs); y_true = np.array(y_true)
    P = predictions_frame_from_arrays(y_true, prob, class_names, ids)
    P.insert(2, "true_name", [class_names[i] for i in y_true]); P.insert(4, "pred_top1_name", [class_names[i] for i in prob.argmax(1)])
    t5 = np.argsort(-prob,1)[:,:5]
    P.insert(5, "top5_idx", [" ".join(map(str,r)) for r in t5]); P.insert(6, "correct_top1", prob.argmax(1)==y_true); P.insert(7, "correct_top5", (t5==y_true[:,None]).any(1))
    P.to_csv(CFG["OUT"]/"predictions"/f"predictions_{split}_{tag}.csv", index=False)
    return P

ALL_METRICS = {}
for s, rd in RUN_DIRS.items():
    P = predict_split(rd/"weights"/"best.pt", SPLIT_DIR, "test", tag=f"yolov8n_cls_seed{s}")
    # group ids aligned by (class, filename):
    key = df[df.split=="test"].set_index(["class_name","filename"]).group_id
    groups = [key.loc[(Path(p).parent.name, Path(p).name)] for p in P.image_id]
    metrics, per_class, cm = evaluate_predictions(P, class_names, groups=groups)
    metrics.update(seed=s, split_manifest_sha256=SPLIT_MANIFEST_SHA, group_source=GROUP_SOURCE, weights=str(rd/"weights"/"best.pt"), gpu_mode=CFG["GPU_MODE"])
    tag = f"yolov8n_cls_seed{s}"
    json.dump(metrics, open(CFG["OUT"]/"metrics"/f"metrics_{tag}.json","w"), indent=2)
    per_class.to_csv(CFG["OUT"]/"metrics"/f"per_class_{tag}.csv", index=False)
    pd.DataFrame(cm, index=class_names, columns=class_names).to_csv(CFG["OUT"]/"metrics"/f"confusion_matrix_{tag}.csv")
    ALL_METRICS[s] = metrics; print(tag, {k: metrics[k] for k in ("top1_accuracy","top5_accuracy","macro_f1","roc_auc_ovr_macro","mAP_macro_stepwise_AP","ece_15bin_top1")})

# mean +/- sample std across training seeds. The split is fixed, so this variation reflects TRAINING randomness only.
keys = ["top1_accuracy","top5_accuracy","macro_precision","macro_recall","macro_f1","weighted_f1","roc_auc_ovr_macro","mAP_macro_stepwise_AP","ece_15bin_top1","brier_multiclass"]
summary = {k: dict(mean=float(np.mean([m[k] for m in ALL_METRICS.values()])), std=float(np.std([m[k] for m in ALL_METRICS.values()], ddof=1)) if len(ALL_METRICS)>1 else None) for k in keys}
json.dump(dict(n_seeds=len(ALL_METRICS), variation_source="training randomness only (single fixed split)", summary=summary), open(CFG["OUT"]/"metrics"/"seed_summary_yolov8n_cls.json","w"), indent=2)
print(json.dumps(summary, indent=2))

WARNING ⚠️ 'half' is deprecated and will be removed in the future. Use 'quantize' instead.
WARNING ⚠️ 'half' is deprecated and will be removed in the future. Use 'quantize' instead.
WARNING ⚠️ 'half' is deprecated and will be removed in the future. Use 'quantize' instead.
WARNING ⚠️ 'half' is deprecated and will be removed in the future. Use 'quantize' instead.
WARNING ⚠️ 'half' is deprecated and will be removed in the future. Use 'quantize' instead.
WARNING ⚠️ 'half' is deprecated and will be removed in the future. Use 'quantize' instead.
WARNING ⚠️ 'half' is deprecated and will be removed in the future. Use 'quantize' instead.
WARNING ⚠️ 'half' is deprecated and will be removed in the future. Use 'quantize' instead.
WARNING ⚠️ 'half' is deprecated and will be removed in the future. Use 'quantize' instead.
WARNING ⚠️ 'half' is deprecated and will be removed in the future. Use 'quantize' instead.
WARNING ⚠️ 'half' is deprecated and will be removed in the future. Use 'quantize' instead.

## 8. Baselines (torchvision) on the SAME split
Only models that are actually trained here may appear in a comparison table. All baselines: ImageNet-pretrained torchvision weights (named below), new 38-way head, all layers fine-tuned, AdamW, cosine schedule over `BASELINE_EPOCHS` epochs, AMP, checkpoint chosen on val top-1, then the same `predict -> evaluate_predictions` path on the test split. **Unavoidable non-equivalences to state in the paper:** torchvision models use ImageNet mean/std normalisation and a different augmentation stack than the Ultralytics pipeline; the optimiser/schedule differ from the YOLO run; epochs are `BASELINE_EPOCHS` (not 30) unless you change it. YOLOv5n-cls is not part of the `ultralytics` package; run it from the YOLOv5 repository (optional cell at the end) or drop it from the comparison.

**Kaggle T4 x2:** baselines are trained on GPU 0 only (`cuda`); GPU 1 is idle in this section. AMP (fp16) is used, which the T4 supports natively.


In [12]:
import torch, torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T
import torchvision.models as tvm

BASELINES = {   # name -> (builder, weights enum name, lr, batch)
    "resnet50":          (tvm.resnet50,          "ResNet50_Weights.IMAGENET1K_V2",            3e-4, 64),
    "vgg16":             (tvm.vgg16,             "VGG16_Weights.IMAGENET1K_V1",               1e-4, 32),
    "mobilenet_v3_large":(tvm.mobilenet_v3_large,"MobileNet_V3_Large_Weights.IMAGENET1K_V2",  3e-4, 64),
    "efficientnet_b0":   (tvm.efficientnet_b0,   "EfficientNet_B0_Weights.IMAGENET1K_V1",     3e-4, 64),
}
def build_baseline(name, n_classes=38):
    builder, wname, _, _ = BASELINES[name]; w = eval("tvm." + wname)
    m = builder(weights=w)
    if name=="resnet50": m.fc = nn.Linear(m.fc.in_features, n_classes)
    elif name=="vgg16": m.classifier[6] = nn.Linear(m.classifier[6].in_features, n_classes)
    else: m.classifier[-1] = nn.Linear(m.classifier[-1].in_features, n_classes)
    return m

MEAN, STD = (0.485,0.456,0.406), (0.229,0.224,0.225)
tf_train = T.Compose([T.RandomResizedCrop(CFG["IMGSZ"], scale=(0.5,1.0)), T.RandomHorizontalFlip(), T.ToTensor(), T.Normalize(MEAN,STD)])
tf_eval  = T.Compose([T.Resize(CFG["IMGSZ"]), T.CenterCrop(CFG["IMGSZ"]), T.ToTensor(), T.Normalize(MEAN,STD)])

class ManifestDS(Dataset):
    def __init__(self, frame, tf): self.f = frame.reset_index(drop=True); self.tf = tf
    def __len__(self): return len(self.f)
    def __getitem__(self, i):
        r = self.f.iloc[i]; return self.tf(Image.open(r.path).convert("RGB")), class_names.index(r.class_name)

@torch.no_grad()
def predict_probs(model, loader, dev):
    model.eval(); out=[]
    for x,_ in loader: out.append(torch.softmax(model(x.to(dev)).float(), 1).cpu().numpy())
    return np.concatenate(out)

def train_baseline(name, seed):
    set_all_seeds(seed); dev = "cuda" if torch.cuda.is_available() else "cpu"
    _,_,lr,bs = BASELINES[name]; m = build_baseline(name).to(dev)
    tr = DataLoader(ManifestDS(df[df.split=="train"], tf_train), batch_size=bs, shuffle=True, num_workers=CFG["WORKERS"], drop_last=True)
    va = DataLoader(ManifestDS(df[df.split=="val"], tf_eval), batch_size=128, num_workers=CFG["WORKERS"])
    opt = torch.optim.AdamW(m.parameters(), lr=lr, weight_decay=1e-4)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=CFG["BASELINE_EPOCHS"]*len(tr))
    try:
        scaler = torch.amp.GradScaler("cuda", enabled=(dev == "cuda"))
    except (AttributeError, TypeError):
        scaler = torch.cuda.amp.GradScaler(enabled=(dev == "cuda"))
    crit = nn.CrossEntropyLoss()
    yv = np.array([class_names.index(c) for c in df[df.split=="val"].class_name]); best, best_state, log = -1, None, []
    for ep in range(CFG["BASELINE_EPOCHS"]):
        m.train(); tl=0; n=0
        for x,y in tr:
            x,y = x.to(dev), y.to(dev); opt.zero_grad(set_to_none=True)
            if dev == "cuda":
                with torch.autocast(device_type="cuda"): loss = crit(m(x), y)
            else:
                loss = crit(m(x), y)
            scaler.scale(loss).backward(); scaler.step(opt); scaler.update(); sched.step(); tl += loss.item()*len(y); n += len(y)
        acc = (predict_probs(m, va, dev).argmax(1)==yv).mean(); log.append(dict(epoch=ep+1, train_loss=tl/n, val_top1=float(acc)))
        if acc>best: best, best_state = acc, {k:v.detach().cpu().clone() for k,v in m.state_dict().items()}
    m.load_state_dict(best_state); pd.DataFrame(log).to_csv(CFG["OUT"]/"runs"/f"{name}_seed{seed}_log.csv", index=False)
    torch.save(best_state, CFG["OUT"]/"runs"/f"{name}_seed{seed}_best.pt")
    # test predictions (test split touched only here, once per final model)
    te = df[df.split=="test"]; dl = DataLoader(ManifestDS(te, tf_eval), batch_size=128, num_workers=CFG["WORKERS"])
    prob = predict_probs(m, dl, dev); y_true = np.array([class_names.index(c) for c in te.class_name])
    P = predictions_frame_from_arrays(y_true, prob, class_names, [str(Path(r.split)/r.class_name/r.filename) for r in te.itertuples()])
    P.insert(2,"true_name",[class_names[i] for i in y_true]); P.insert(4,"pred_top1_name",[class_names[i] for i in prob.argmax(1)])
    t5=np.argsort(-prob,1)[:,:5]; P.insert(5,"top5_idx",[" ".join(map(str,r)) for r in t5]); P.insert(6,"correct_top1",prob.argmax(1)==y_true); P.insert(7,"correct_top5",(t5==y_true[:,None]).any(1))
    tag=f"{name}_seed{seed}"; P.to_csv(CFG["OUT"]/"predictions"/f"predictions_test_{tag}.csv", index=False)
    met, pc, cm = evaluate_predictions(P, class_names); met.update(seed=seed, split_manifest_sha256=SPLIT_MANIFEST_SHA)
    json.dump(met, open(CFG["OUT"]/"metrics"/f"metrics_{tag}.json","w"), indent=2); pc.to_csv(CFG["OUT"]/"metrics"/f"per_class_{tag}.csv", index=False)
    return met

BASELINE_METRICS = {}
for name in BASELINES:
    for s in CFG["TRAIN_SEEDS"][:1]:          # increase to all seeds if time permits; report how many were run
        BASELINE_METRICS[(name,s)] = train_baseline(name, s); print(name, s, BASELINE_METRICS[(name,s)]["top1_accuracy"])

Downloading: "https://download.pytorch.org/models/resnet50-11ad3fa6.pth" to /root/.cache/torch/hub/checkpoints/resnet50-11ad3fa6.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 251MB/s]


resnet50 0 0.9979212680265038
Downloading: "https://download.pytorch.org/models/vgg16-397923af.pth" to /root/.cache/torch/hub/checkpoints/vgg16-397923af.pth


100%|██████████| 528M/528M [00:02<00:00, 226MB/s] 


vgg16 0 0.993893724827855
Downloading: "https://download.pytorch.org/models/mobilenet_v3_large-5c1a4163.pth" to /root/.cache/torch/hub/checkpoints/mobilenet_v3_large-5c1a4163.pth


100%|██████████| 21.1M/21.1M [00:00<00:00, 164MB/s]


mobilenet_v3_large 0 0.9977913472781603
Downloading: "https://download.pytorch.org/models/efficientnet_b0_rwightman-7f5810bc.pth" to /root/.cache/torch/hub/checkpoints/efficientnet_b0_rwightman-7f5810bc.pth


100%|██████████| 20.5M/20.5M [00:00<00:00, 196MB/s]


efficientnet_b0 0 0.9977913472781603


In [13]:
# OPTIONAL - YOLOv5n-cls via the original YOLOv5 repository (not executed; requires a git clone with network access).
# !git clone https://github.com/ultralytics/yolov5 && pip install -r yolov5/requirements.txt
# !python yolov5/classify/train.py --model yolov5n-cls.pt --data {SPLIT_DIR} --epochs {CFG["BASELINE_EPOCHS"]} --img 224 --batch-size 64 --seed 0 --project {CFG["OUT"]/"runs"} --name yolov5n_cls_seed0
# Then produce predictions with the same schema as Section 7 and call evaluate_predictions. If this is not run, DELETE YOLOv5n-cls from the manuscript.

## 9. Export, numerical parity and measured benchmarks
Each row of the benchmark table records: artefact/format, device, software versions, batch size, precision, warm-up and timed iterations, synchronisation, whether pre/post-processing is included, mean / median / p95 / p99 latency, throughput and (where measurable) peak memory. "model_only" = raw module or ONNX session on a ready tensor; "wrapper" = `ultralytics` `predict` call on a ready tensor; "end_to_end" = JPEG decode + resize + tensor + model + softmax/top-5 for one image. Backends that cannot run are recorded as `UNAVAILABLE: <reason>`, never filled with literature values.

**Kaggle T4 x2:** all GPU benchmark rows are measured on GPU 0 (`ENV['gpu']`) with nothing else running on it. TensorRT needs the `tensorrt` pip package; without it those rows are recorded as `UNAVAILABLE`. A TensorRT engine is tied to the GPU model and TensorRT version it was built with.


In [14]:
from ultralytics.utils.torch_utils import get_flops
import torchvision.transforms as T
BEST = RUN_DIRS[CFG["TRAIN_SEEDS"][0]] / "weights" / "best.pt"
if not BEST.is_file(): raise FileNotFoundError(f"Trained checkpoint not found: {BEST}. Complete Section 6 first.")
ym = YOLO(str(BEST))
n_params_unfused = sum(p.numel() for p in ym.model.parameters())
ym_f = YOLO(str(BEST)); ym_f.fuse()
n_params_fused = sum(p.numel() for p in ym_f.model.parameters())
onnx_path = Path(ym.export(format="onnx", imgsz=CFG["IMGSZ"], simplify=True, dynamic=False, half=False, batch=1))
size_info = dict(params_unfused=n_params_unfused, params_fused=n_params_fused, gflops_224=float(get_flops(ym.model, CFG["IMGSZ"])),
                 best_pt_MB=os.path.getsize(BEST)/1e6, onnx_MB=os.path.getsize(onnx_path)/1e6, onnx_path=str(onnx_path))
json.dump(size_info, open(CFG["OUT"]/"benchmarks"/"model_size_info.json","w"), indent=2); print(size_info)

def load_tensor(path, size=CFG["IMGSZ"]):
    im = Image.open(path).convert("RGB"); im = T.Resize(size)(im); im = T.CenterCrop(size)(im)
    return torch.from_numpy(np.asarray(im, dtype=np.float32)/255.).permute(2,0,1)       # 0-1 scaling, no mean/std (Ultralytics classify default)

_test_pool = df[df.split == "test"].path
if _test_pool.empty: raise ValueError("Test split is empty; run the dataset audit and split cells first.")
test_paths = _test_pool.sample(n=min(256, len(_test_pool)), random_state=0).tolist()
X = torch.stack([load_tensor(p) for p in test_paths])

# --- parity: PyTorch vs ONNX Runtime on identical tensors -------------------------------------
import onnxruntime as ort
sess = ort.InferenceSession(str(onnx_path), providers=["CPUExecutionProvider"]); inp = sess.get_inputs()[0].name
mod = ym.model.eval().float().cpu()
with torch.inference_mode():
    o = mod(X); o = o[0] if isinstance(o, (tuple, list)) else o
pt_probs = o.cpu().numpy()
ox_probs = np.concatenate([sess.run(None, {inp: X[i:i+1].numpy()})[0] for i in range(len(X))])
parity = dict(n=len(X), max_abs_diff=float(np.abs(pt_probs-ox_probs).max()), top1_agreement=float((pt_probs.argmax(1)==ox_probs.argmax(1)).mean()))
json.dump(parity, open(CFG["OUT"]/"benchmarks"/"onnx_parity.json","w"), indent=2); print(parity)

YOLOv8n-cls summary (fused): 30 layers, 1,483,558 parameters, 0 gradients, 3.3 GFLOPs
WARNING ⚠️ 'half' is deprecated and will be removed in the future. Use 'quantize' instead.
Ultralytics 8.4.189 🚀 Python-3.13.15 torch-2.11.0+cu128 CPU (Intel Xeon CPU @ 2.00GHz)
💡 ProTip: Export to OpenVINO format for best performance on Intel hardware. Learn more at https://docs.ultralytics.com/integrations/openvino
YOLOv8n-cls summary (fused): 30 layers, 1,483,558 parameters, 0 gradients, 0.4 GFLOPs

PyTorch: starting from '/kaggle/working/experiments/runs/yolov8n_cls_seed0/weights/best.pt' with input shape (1, 3, 224, 224) BCHW and output shape(s) (1, 38) (2.9 MB)

ONNX: starting export with onnx 1.22.0 opset 18...
ONNX: slimming with onnxslim 0.1.98...
ONNX: export success ✅ 0.6s, saved as '/kaggle/working/experiments/runs/yolov8n_cls_seed0/weights/best.onnx' (5.7 MB)

Export complete (0.9s)
Results saved to /kaggle/working/experiments/runs/yolov8n_cls_seed0/weights/best.onnx
Predict:         yolo

In [15]:
def time_callable(fn, warmup=50, iters=500, cuda=False):
    """Time inference calls, excluding autograd graph construction."""
    with torch.inference_mode():
        for _ in range(warmup): fn()
        if cuda: torch.cuda.synchronize()
        ts = []
        for _ in range(iters):
            if cuda: torch.cuda.synchronize()
            t0 = time.perf_counter_ns(); fn()
            if cuda: torch.cuda.synchronize()
            ts.append((time.perf_counter_ns() - t0) / 1e6)
    return np.array(ts)

def stats_row(ts, bs, **meta):
    return dict(**meta, batch=bs, warmup=50, iters=len(ts), mean_ms=ts.mean(), median_ms=np.median(ts), p95_ms=np.percentile(ts,95), p99_ms=np.percentile(ts,99), std_ms=ts.std(ddof=1),
                throughput_img_s=bs/(ts.mean()/1e3))

ROWS = []; RAW = {}
def add(label, fn, bs, cuda=False, **meta):
    try:
        if cuda: torch.cuda.reset_peak_memory_stats()
        ts = time_callable(fn, cuda=cuda); r = stats_row(ts, bs, backend=label, **meta)
        # For ONNX Runtime and TensorRT, PyTorch's allocator cannot see all backend allocations.
        r["peak_cuda_mem_MiB"] = (torch.cuda.max_memory_allocated() / 2**20) if (cuda and label.startswith("pytorch_")) else None
        r["memory_note"] = "PyTorch CUDA allocator only; backend allocations may be excluded" if cuda else "not measured"
        ROWS.append(r); RAW[f"{label}_b{bs}"] = ts
    except Exception as e:
        ROWS.append(dict(backend=label, batch=bs, status=f"UNAVAILABLE: {type(e).__name__}: {e}", **meta))

HW = ENV.get("gpu","CPU only")
def _ort_cuda_ok():
    """ONNX Runtime may silently fall back to CPU when CUDA/cuDNN libs do not match - verify the provider that is really in use."""
    try: return "CUDAExecutionProvider" in ort.InferenceSession(str(onnx_path), providers=["CUDAExecutionProvider"]).get_providers()
    except Exception: return False
ORT_CUDA_OK = _ort_cuda_ok()
for bs in (1, 32):
    xb = X[:bs]
    m_cpu = ym.model.eval().float().cpu()
    add("pytorch_cpu_fp32", lambda: m_cpu(xb), bs, artefact="best.pt", path="model_only", precision="fp32", hw=ENV["cpu"])
    if torch.cuda.is_available():
        m_gpu = YOLO(str(BEST)).model.eval().float().cuda(); xg = xb.cuda()
        with torch.no_grad(): add("pytorch_cuda_fp32", lambda: m_gpu(xg), bs, cuda=True, artefact="best.pt", path="model_only", precision="fp32", hw=HW)
    s_cpu = ort.InferenceSession(str(onnx_path), providers=["CPUExecutionProvider"]); xn = xb.numpy()
    if bs==1: add("onnxruntime_cpu_fp32", lambda: s_cpu.run(None, {inp: xn}), bs, artefact=onnx_path.name, path="model_only", precision="fp32", hw=ENV["cpu"])
    if ORT_CUDA_OK:
        s_gpu = ort.InferenceSession(str(onnx_path), providers=["CUDAExecutionProvider"])
        add("onnxruntime_cuda_fp32", lambda: s_gpu.run(None, {inp: xn}), bs, cuda=True, artefact=onnx_path.name, path="model_only", precision="fp32", hw=HW)
    else: ROWS.append(dict(backend="onnxruntime_cuda_fp32", batch=bs, status="UNAVAILABLE: CUDAExecutionProvider not available or session fell back to CPU"))

# TensorRT through Ultralytics (wrapper path - includes Ultralytics pre/post-processing; label it as such)
for half in (False, True):
    try:
        eng = Path(YOLO(str(BEST)).export(format="engine", imgsz=CFG["IMGSZ"], half=half, batch=1)); tm = YOLO(str(eng)); x1 = X[:1].cuda()
        add(f"tensorrt_{'fp16' if half else 'fp32'}_ultralytics_wrapper", lambda: tm.predict(x1, imgsz=CFG["IMGSZ"], verbose=False), 1, cuda=True, artefact=eng.name, path="wrapper", precision="fp16" if half else "fp32", hw=HW)
    except Exception as e:
        ROWS.append(dict(backend=f"tensorrt_{'fp16' if half else 'fp32'}", batch=1, status=f"UNAVAILABLE: {type(e).__name__}: {e}"))

# end-to-end (single image, file -> top-5) on ONNX Runtime CPU
def e2e(p=test_paths[0]):
    x = load_tensor(p).unsqueeze(0).numpy(); y = s_cpu.run(None, {inp: x})[0]; return np.argsort(-y[0])[:5]
add("onnxruntime_cpu_fp32_end_to_end", e2e, 1, artefact=onnx_path.name, path="end_to_end(decode+resize+model+top5)", precision="fp32", hw=ENV["cpu"])

bench = pd.DataFrame(ROWS); bench.to_csv(CFG["OUT"]/"benchmarks"/"benchmark_results.csv", index=False)
np.savez_compressed(CFG["OUT"]/"benchmarks"/"latency_raw_ms.npz", **RAW); print(bench.to_string())

2026-10-25 20:12:28.312155741 [E:onnxruntime:Default, provider_bridge_ort.cc:2458 operator()] /onnxruntime_src/onnxruntime/core/session/provider_bridge_ort.cc:2043 onnxruntime::Provider& onnxruntime::ProviderLibrary::Get() [ONNXRuntimeError] : 1 : FAIL : Failed to load library /usr/local/lib/python3.13/dist-packages/onnxruntime/capi/libonnxruntime_providers_cuda.so with error: libcublasLt.so.13: cannot open shared object file: No such file or directory



WARNING ⚠️ 'half' is deprecated and will be removed in the future. Use 'quantize' instead.
WARNING ⚠️ TensorRT requires GPU export, automatically assigning device=0
Ultralytics 8.4.189 🚀 Python-3.13.15 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
YOLOv8n-cls summary (fused): 30 layers, 1,483,558 parameters, 0 gradients, 0.4 GFLOPs

PyTorch: starting from '/kaggle/working/experiments/runs/yolov8n_cls_seed0/weights/best.pt' with input shape (1, 3, 224, 224) BCHW and output shape(s) (1, 38) (2.9 MB)

ONNX: starting export with onnx 1.22.0 opset 18...
ONNX: slimming with onnxslim 0.1.98...
ONNX: export success ✅ 0.3s, saved as '/kaggle/working/experiments/runs/yolov8n_cls_seed0/weights/best.onnx' (5.7 MB)
requirements: Ultralytics requirement ['tensorrt-cu12>=7.0.0,!=10.2.0'] not found, attempting AutoUpdate...
Using Python 3.13.15 environment at: /usr
Resolved 3 packages in 159ms
Prepared 3 packages in 1m 07s
Installed 3 packages in 9ms
 + tensorrt-cu12==11.4.0.106
 + tensorrt-cu12-bind

## 10. External field-image validation (PlantDoc or another documented dataset)
Before running: (1) confirm the dataset's provenance, licence and label list on its official page; (2) fill `LABEL_MAP` below by hand with a justified mapping from external labels to the 38 class names; (3) every external image whose label is not mapped is **excluded and counted**, never scored as correct. The empty dictionary makes the cell stop until the mapping has been written.

In [16]:
EXT_ROOT = Path(os.environ.get("EXT_ROOT", "/kaggle/input/plantdoc-test"))  # expected: <external-class>/<image>
LABEL_MAP = {}  # Authors must fill and justify, e.g. {"Tomato leaf late blight": "Tomato___Late_blight"}
STATUS_PATH = CFG["OUT"] / "external" / "external_status.json"

if not LABEL_MAP:
    external_status = {"status": "NOT RUN", "reason": "LABEL_MAP is empty; authors must define and justify external-label to PlantVillage-class mappings.", "external_root": str(EXT_ROOT)}
    json.dump(external_status, open(STATUS_PATH, "w"), indent=2)
    print(external_status)
elif not EXT_ROOT.is_dir():
    external_status = {"status": "NOT RUN", "reason": "External dataset directory not found.", "external_root": str(EXT_ROOT)}
    json.dump(external_status, open(STATUS_PATH, "w"), indent=2)
    print(external_status)
else:
    if not all(v in class_names for v in LABEL_MAP.values()):
        raise ValueError("Every LABEL_MAP target must exactly match one of class_names; see audit/class_mapping.json")
    ext = pd.DataFrame([dict(path=str(f), ext_label=f.parent.name) for f in EXT_ROOT.rglob("*.*")
                        if f.suffix.lower() in (".jpg", ".jpeg", ".png")])
    if ext.empty:
        external_status = {"status": "NOT RUN", "reason": "No supported images found under external dataset directory.", "external_root": str(EXT_ROOT)}
        json.dump(external_status, open(STATUS_PATH, "w"), indent=2)
        print(external_status)
    else:
        ext["mapped"] = ext.ext_label.isin(LABEL_MAP)
        ext["true_name"] = ext.ext_label.map(LABEL_MAP)
        print("external images:", len(ext), "| mapped:", int(ext.mapped.sum()), "| excluded (unmapped):", int((~ext.mapped).sum()))
        print("unmapped labels:", sorted(ext[~ext.mapped].ext_label.unique()), "| mapped target classes:", int(ext[ext.mapped].true_name.nunique()))
        em = ext[ext.mapped].reset_index(drop=True)
        if em.empty:
            external_status = {"status": "NOT RUN", "reason": "No external labels matched LABEL_MAP; no accuracy was calculated.", "n_total": int(len(ext)), "n_mapped": 0, "n_excluded": int(len(ext))}
            json.dump(external_status, open(STATUS_PATH, "w"), indent=2)
            print(external_status)
        else:
            res = ym.predict(em.path.tolist(), imgsz=CFG["IMGSZ"], batch=64, verbose=False)
            prob = np.stack([r.probs.data.float().cpu().numpy() for r in res])
            y = np.array([class_names.index(n) for n in em.true_name])
            ext_out = dict(status="EXECUTED", n_total=int(len(ext)), n_mapped=int(len(em)), n_excluded=int((~ext.mapped).sum()),
                           n_target_classes=int(em.true_name.nunique()),
                           top1_over_38_outputs=float((prob.argmax(1) == y).mean()),
                           top5_over_38_outputs=float((np.argsort(-prob, 1)[:, :5] == y[:, None]).any(1).mean()))
            em["pred_name"] = [class_names[i] for i in prob.argmax(1)]
            em.to_csv(CFG["OUT"] / "external" / "external_predictions.csv", index=False)
            json.dump(dict(label_map=LABEL_MAP, **ext_out), open(CFG["OUT"] / "external" / "external_metrics.json", "w"), indent=2)
            json.dump({"status": "EXECUTED", "external_root": str(EXT_ROOT)}, open(STATUS_PATH, "w"), indent=2)
            print(ext_out)


{'status': 'NOT RUN', 'reason': 'LABEL_MAP is empty; authors must define and justify external-label to PlantVillage-class mappings.', 'external_root': '/kaggle/input/plantdoc-test'}


## 11. Figures from saved artefacts only (PNG + PDF)

In [17]:
import matplotlib; import matplotlib.pyplot as plt
plt.rcParams.update({"font.family":"DejaVu Serif","font.size":9,"savefig.dpi":400,"axes.spines.top":False,"axes.spines.right":False})
FD = CFG["OUT"]/"figures"
def save(fig, name):
    for e in ("png","pdf"): fig.savefig(FD/f"{name}.{e}", bbox_inches="tight")

# (a) sample grid: one seeded random image per class, from the variant actually used
rng = np.random.default_rng(0); fig, axs = plt.subplots(5, 8, figsize=(10, 6.5))
for ax in axs.ravel(): ax.axis("off")
for k, c in enumerate(class_names):
    r = df[df.class_name==c].sample(1, random_state=int(rng.integers(1e6))).iloc[0]; ax = axs.ravel()[k]
    ax.imshow(Image.open(r.path).convert("RGB")); ax.set_title(c.replace("___","\n").replace("_"," ")[:28], fontsize=4.5)
fig.suptitle(f"Fig. 1  Representative images, one per class (PlantVillage '{CFG['VARIANT']}' variant)", fontsize=9); save(fig, "fig1_samples"); plt.close(fig)

# (b) training curves of seed 0 from results.csv (includes validation loss)
rc = pd.read_csv(RUN_DIRS[CFG["TRAIN_SEEDS"][0]]/"results.csv"); rc.columns = rc.columns.str.strip()
fig, ax = plt.subplots(1, 2, figsize=(7.2, 2.9))
ax[0].plot(rc.epoch, rc["train/loss"], label="train"); ax[0].plot(rc.epoch, rc["val/loss"], "--", label="val"); ax[0].set(xlabel="Epoch", ylabel="Cross-entropy loss"); ax[0].legend(frameon=False)
ax[1].plot(rc.epoch, 100*rc["metrics/accuracy_top1"], label="top-1"); ax[1].plot(rc.epoch, 100*rc["metrics/accuracy_top5"], "--", label="top-5"); ax[1].set(xlabel="Epoch", ylabel="Validation accuracy (%)"); ax[1].legend(frameon=False)
fig.tight_layout(); save(fig, "fig2_training_curves"); plt.close(fig)

# (c) confusion matrix and per-class F1 from the saved files (seed 0)
tag = f"yolov8n_cls_seed{CFG['TRAIN_SEEDS'][0]}"
cmf = pd.read_csv(CFG["OUT"]/"metrics"/f"confusion_matrix_{tag}.csv", index_col=0); pcf = pd.read_csv(CFG["OUT"]/"metrics"/f"per_class_{tag}.csv")
_row_sums = cmf.values.sum(1, keepdims=True)
cmn = np.divide(cmf.values, _row_sums, out=np.zeros_like(cmf.values, dtype=float), where=_row_sums != 0)
fig, ax = plt.subplots(figsize=(9,8)); im = ax.imshow(cmn, cmap="Blues", vmin=0, vmax=1); plt.colorbar(im, fraction=0.04, label="Row-normalised count")
ax.set_xticks(range(38)); ax.set_yticks(range(38)); ax.set_xticklabels(range(38), fontsize=5); ax.set_yticklabels([f"{i}  {n[:26]}" for i,n in enumerate(class_names)], fontsize=5)
ax.set(xlabel="Predicted class index", ylabel="True class"); save(fig, "fig3_confusion_matrix"); plt.close(fig)
fig, ax = plt.subplots(figsize=(6,7)); o = pcf.sort_values("f1"); ax.barh([n[:30] for n in o.class_name], o.f1); ax.set(xlabel="F1 (test)", xlim=(max(0, o.f1.min()-0.02), 1.005)); ax.tick_params(axis="y", labelsize=5)
save(fig, "fig3b_per_class_f1"); plt.close(fig)

# (d) accuracy vs measured per-image latency; only real successful benchmark rows are used.
bench_path = CFG["OUT"] / "benchmarks" / "benchmark_results.csv"
metric_path = CFG["OUT"] / "metrics" / f"metrics_yolov8n_cls_seed{CFG['TRAIN_SEEDS'][0]}.json"
if bench_path.is_file() and metric_path.is_file():
    b = pd.read_csv(bench_path)
    if "status" in b: b = b[b["status"].isna()]
    if {"mean_ms", "batch", "backend"}.issubset(b.columns): b = b.dropna(subset=["mean_ms", "batch"])
    else: b = pd.DataFrame()
    if not b.empty:
        met = json.load(open(metric_path))
        b = b.copy(); b["latency_ms_per_image"] = b["mean_ms"] / b["batch"]
        fig, ax = plt.subplots(figsize=(7, 4))
        for _, row in b.iterrows():
            ax.scatter(row["latency_ms_per_image"], 100 * met["top1_accuracy"])
            ax.annotate(str(row["backend"]), (row["latency_ms_per_image"], 100 * met["top1_accuracy"]), fontsize=6, xytext=(3, 3), textcoords="offset points")
        ax.set(xlabel="Measured mean latency (ms/image; batch mean divided by batch size)", ylabel="Held-out Top-1 accuracy (%)")
        ax.set_title("YOLOv8n-cls accuracy vs measured backend latency")
        save(fig, "fig4_accuracy_latency"); plt.close(fig)
        print("Saved accuracy-vs-latency figure from", len(b), "successful measured benchmark rows")
    else:
        print("Accuracy-vs-latency figure not produced: no successful measured benchmark rows are available.")
else:
    print("Accuracy-vs-latency figure not produced: run benchmark cell and held-out evaluation first.")


Saved accuracy-vs-latency figure from 8 successful measured benchmark rows


## 12. Artefact manifest (provenance of every generated file)

In [18]:
manifest = []
for p in sorted(CFG["OUT"].rglob("*")):
    if p.is_file() and not p.is_symlink() and p.name != "artifact_manifest.json" and "ultralytics_layout" not in p.parts and p.suffix in (".csv", ".json", ".png", ".pdf", ".npz", ".onnx", ".pt", ".yaml", ".engine"):
        manifest.append(dict(path=str(p.relative_to(CFG["OUT"])), bytes=p.stat().st_size, sha256=sha256_file(p)))
json.dump(dict(environment=ENV, split_manifest_sha256=SPLIT_MANIFEST_SHA, group_source=GROUP_SOURCE, files=manifest), open(CFG["OUT"]/"artifact_manifest.json","w"), indent=2)
print(len(manifest), "files recorded")

80 files recorded
